# Notebook 018: MapType Creation, Key Extraction and Dynamic Properties

**Module:** Module 02: Complex Data Types & Nested Structures Handling  
**Difficulty:** `Intermediate`  
**Databricks Masterclass Reference:** Video 14: Master Unity Catalog & Video 23: Interview Questions  

---

## 1. Business Scenario & Real-World Context
An advertising technology platform captures custom user ad-tracking attributes (cookies, campaign IDs, UTM tags, experiment variants) as dynamic key-value pairs where keys vary across partners and cannot be fixed into a static schema.

## 2. Technical Requirement & Problem Statement
Master MapType operations: construct maps from column pairs (`create_map()`), extract keys (`map_keys()`), extract values (`map_values()`), build maps from paired arrays (`map_from_arrays()`), and query dynamic attributes safely without null pointer exceptions.

## 3. Spark & Databricks Architecture Notes
MapType represents key-value collections with arbitrary keys of identical data type. Unlike StructType (where field names are fixed in schema metadata), MapType allows dynamic runtime keys, making it ideal for sparse key-value attributes.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_018_MapType_Creation,_Key_Extraction_and_Dynamic_Properties")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Creating Ad-Tech Campaign Metadata with MapType
Construct maps using create_map and map_from_arrays.

In [ ]:
# Build map using paired key-value lists
campaign_data = [
    ("EV-1", ["utm_source", "utm_campaign", "device"], ["google", "black_friday", "mobile"]),
    ("EV-2", ["utm_source", "experiment_group"], ["facebook", "variant_b"]),
    ("EV-3", ["utm_source", "utm_medium", "utm_campaign"], ["linkedin", "cpc", "q4_enterprise"])
]

df_raw_campaigns = spark.createDataFrame(campaign_data, ["event_id", "param_keys", "param_vals"])

# Convert parallel arrays into MapType
df_mapped = df_raw_campaigns.withColumn(
    "attributes",
    F.map_from_arrays(F.col("param_keys"), F.col("param_vals"))
)

df_mapped.printSchema()
df_mapped.select("event_id", "attributes").show(truncate=False)

## 6. Querying Map Keys, Values & Dynamic Lookup
Access map values by key name using bracket notation and inspect key catalogs.

In [ ]:
# Querying specific keys
df_extracted = df_mapped.withColumn(
    "source", F.col("attributes")["utm_source"]
).withColumn(
    "campaign", F.coalesce(F.col("attributes")["utm_campaign"], F.lit("ORGANIC"))
).withColumn(
    "all_keys", F.map_keys(F.col("attributes"))
).withColumn(
    "all_values", F.map_values(F.col("attributes"))
)

df_extracted.select("event_id", "source", "campaign", "all_keys").show(truncate=False)

## 7. Exploding MapType into Relational Key-Value Rows
Convert sparse key-value pairs into normalized relational rows.

In [ ]:
df_kv_pairs = df_mapped.select(
    "event_id",
    F.explode("attributes").alias("attr_key", "attr_value")
)

df_kv_pairs.show()

## 9. Key Architectural Takeaways
- **Use `MapType` for semi-structured data where keys are dynamic, variable, or unpredictable.**
- **`map_from_arrays(keys_col, values_col)` builds maps from two parallel array columns.**
- **Accessing non-existent map keys returns `NULL` safely rather than raising runtime errors.**

## 10. Senior Data Engineering Interview Questions
1. What is the difference between `MapType` and `StructType` in PySpark?
2. When should you choose `MapType` over a normalized child table in Lakehouse architecture?
3. How does `explode(map_col)` generate key and value columns simultaneously?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
